# A short presentation showing speedup when switching from pandas to FireDucks

🔥 🐦[**FireDucks**](https://fireducks-dev.github.io/) is a high-performance compiler-accelerated DataFrame library with highly compatible pandas APIs developed to speedup a pandas application without any manual source code changes. It comes with a multi-threaded C++ kernel and automatic query optimization features (powered by an in-built compiler) with lazy-execution model.

In this test drive, we will be using [ Parking Violations Issued - Fiscal Year 2022](https://data.cityofnewyork.us/City-Government/Parking-Violations-Issued-Fiscal-Year-2022/7mxj-7a6y/about_data) dataset from NYC Open Data.

REF: https://colab.research.google.com/drive/12tCzP94zFG2BRduACucn5Q_OcX1TUKY3




In [1]:
!pip install fireducks


[notice] A new release of pip is available: 23.3.1 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
# os.environ["FIREDUCKS_FLAGS"] = "--benchmark-mode"
os.environ["FIRE_LOG_LEVEL"] = "3"

In [3]:
%load_ext fireducks.pandas
import time
import pandas as pd

In [4]:
print(f"evaluation with {pd.__name__}")

evaluation with fireducks.pandas


In [5]:
## Let's load the parquet dataset

In [6]:
# Data can be downloaded from here:
#!wget https://data.rapids.ai/datasets/nyc_parking/nyc_parking_violations_2022.parquet

In [7]:
# this method enforces the execution of the compiled IRs for the input frame.
# use it for FireDucks to measure the actual computation time
def evaluate(df):
  if hasattr(df, "_evaluate"):
    df._evaluate()

In [8]:
t0 = time.time()
df = pd.read_parquet(
    "nyc_parking_violations_2022.parquet",
    columns=["Registration State", "Violation Description",
             "Vehicle Body Type", "Issue Date", "Summons Number"]
)
evaluate(df)
load_t = time.time() - t0
df

2024-11-21 17:03:03.085575: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main() {
  %t0 = read_parquet('nyc_parking_violations_2022.parquet', ['Registration State', 'Violation Description', 'Vehicle Body Type', 'Issue Date', 'Summons Number'])
  return(%t0)
}
2024-11-21 17:03:03.086294: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @main() {
  %t0 = read_parquet('nyc_parking_violations_2022.parquet', ['Registration State', 'Violation Description', 'Vehicle Body Type', 'Issue Date', 'Summons Number'])
  return(%t0)
}
2024-11-21 17:03:03.371269: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !table) {
  %v1 = get_shape(%arg0)
  return(%v1)
}
2024-11-21 17:03:03.371991: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @main(%arg0: !table) {
  %v1 = get_shape(%arg0)
  return(%v1)
}
2024-11-21 17:03:03.373000: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !table) {
  %t1 = slice(%arg0, 0, 6, 1)


,Registration State,Violation Description,Vehicle Body Type,Issue Date,Summons Number
0,NY,<NA>,VAN,06/25/2021,1457617912
1,NY,<NA>,SUBN,06/25/2021,1457617924
2,TX,<NA>,SDN,06/17/2021,1457622427
3,MO,<NA>,SDN,06/16/2021,1457638629
4,NY,<NA>,TAXI,07/04/2021,1457639580
...,...,...,...,...,...
15435602,99,21-No Parking (street clean),SUBN,06/07/2022,8995222761
15435603,TN,21-No Parking (street clean),PICK,06/07/2022,8995222773
15435604,NY,21-No Parking (street clean),2DSD,06/07/2022,8995222785
15435605,VA,21-No Parking (street clean),SUBN,06/07/2022,8995222827


## Q1: Which parking violation is most commonly committed by vehicles from various U.S states?

In [9]:
t1 = time.time()
r1 = (df[["Registration State", "Violation Description"]]
 .value_counts()
 .groupby("Registration State")
 .head(1)
 .sort_index()
 .reset_index()
)
evaluate(r1)
q1_t = time.time() - t1
r1

2024-11-21 17:03:03.387744: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !table) {
  %t1 = project(%arg0, ['Registration State', 'Violation Description'])
  %t2 = value_counts(%t1)
  %v3, %v4 = to_pandas.frame.metadata(%t2)
  return(%t2, %v3, %v4)
}
2024-11-21 17:03:03.388206: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @main(%arg0: !table) {
  %t1 = project(%arg0, ['Registration State', 'Violation Description'])
  %t2 = value_counts(%t1)
  %v3, %v4 = to_pandas.frame.metadata(%t2)
  return(%t2, %v3, %v4)
}
2024-11-21 17:03:03.450827: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !pyobj, %arg1: !metadata) {
  %t2 = from_pandas.frame.metadata(%arg0, %arg1)
  %v3 = make_vector_or_scalar_of_scalar.from_scalar(<<UNSUPPORTED SCALAR: fireducks.make_scalar.i1>>)
  %t4 = sort_index(%t2, %v3)
  %t5 = reset_index(%t4)
  return(%t5)
}
2024-11-21 17:03:03.451633: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @m

,Registration State,Violation Description,0
0,99,<NA>,17550
1,AB,14-No Standing,22
2,AK,PHTO SCHOOL ZN SPEED VIOLATION,125
3,AL,PHTO SCHOOL ZN SPEED VIOLATION,3668
4,AR,PHTO SCHOOL ZN SPEED VIOLATION,537
...,...,...,...
62,VT,PHTO SCHOOL ZN SPEED VIOLATION,3024
63,WA,21-No Parking (street clean),3732
64,WI,14-No Standing,1639
65,WV,PHTO SCHOOL ZN SPEED VIOLATION,1185


## Q2: Which vehicle body types are most frequently involved in parking violations?

In [10]:
t2 = time.time()
r2 = (df
 .groupby(["Vehicle Body Type"])
 .agg({"Summons Number": "count"})
 .rename(columns={"Summons Number": "Count"})
 .sort_values(["Count"], ascending=False)
)
evaluate(r2)
q2_t = time.time() - t2
r2

2024-11-21 17:03:03.474847: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !table) {
  %t1 = groupby_agg(%arg0, ['Vehicle Body Type'], ['count'], ['Summons Number'], [])
  %t2 = rename_specified(%t1, ['Summons Number'], ['Count'])
  %t3 = sort_values(%t2, ['Count'], [0])
  return(%t3)
}
2024-11-21 17:03:03.475397: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @main(%arg0: !table) {
  %t1 = groupby_agg(%arg0, ['Vehicle Body Type'], ['count'], ['Summons Number'], [])
  %t2 = rename_specified(%t1, ['Summons Number'], ['Count'])
  %t3 = sort_values(%t2, ['Count'], [0])
  return(%t3)
}
2024-11-21 17:03:03.496947: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !table) {
  %v1 = get_shape(%arg0)
  return(%v1)
}
2024-11-21 17:03:03.497554: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @main(%arg0: !table) {
  %v1 = get_shape(%arg0)
  return(%v1)
}
2024-11-21 17:03:03.498352: 2818307 fireducks/lib/fireducks_core

,Count
Vehicle Body Type,
SUBN,6449007
4DSD,4402991
VAN,1317899
DELV,436430
PICK,429798
...,...
YANT,1
YBSD,1
YEL,1


## Q3. How do parking violations vary across days of the week?

In [11]:
t3 = time.time()
weekday_names = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday",
}

df["Issue Date"] = df["Issue Date"].astype("datetime64[ms]")
df["issue_weekday"] = df["Issue Date"].dt.weekday.map(weekday_names)
r3 = df.groupby(["issue_weekday"])["Summons Number"].count().sort_values()
evaluate(r3)
q3_t = time.time() - t3
r3

2024-11-21 17:03:03.511043: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !table, %arg1: !pyobj, %arg2: !metadata) {
  %t3 = project(%arg0, 'Issue Date')
  %t4 = cast(%t3, [], ['datetime64[ms]'])
  %t5 = setitem(%arg0, 'Issue Date', %t4)
  %t6 = project(%t5, 'Issue Date')
  %t7 = datetime_extract(%t6, 'day_of_week')
  %t8 = from_pandas.frame.metadata(%arg1, %arg2)
  %t9 = column_dict_map(%t7, %t8)
  %t10 = setitem(%t5, 'issue_weekday', %t9)
  %t11 = groupby_select_agg(%t10, ['issue_weekday'], ['count'], [], [], 'Summons Number')
  %t12 = sort_values(%t11, [], [1])
  return(%t10, %t12)
}


2024-11-21 17:03:03.511702: 2818307 fireducks/passes/pat/pat.cc:90] Optimize GroupbySelectAggSortPat


2024-11-21 17:03:03.511702: 2818307 fireducks/passes/pat/pat.cc:90] Optimize GroupbySelectAggSortPat
2024-11-21 17:03:03.511839: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @main(%arg0: !table, %arg1: !pyobj, %arg2: !metadata) {
  %t3 = project(%arg0, 'Issue Date')
  %t4 = cast(%t3, [], ['datetime64[ms]'])
  %t5 = setitem(%arg0, 'Issue Date', %t4)
  %t6 = project(%t5, 'Issue Date')
  %t7 = datetime_extract(%t6, 'day_of_week')
  %t8 = from_pandas.frame.metadata(%arg1, %arg2)
  %t9 = column_dict_map(%t7, %t8)
  %t10 = setitem(%t5, 'issue_weekday', %t9)
  %t11 = groupby_select_agg(%t10, ['issue_weekday'], ['count'], [], [], 'Summons Number')
  %t12 = sort_values(%t11, [], [1])
  return(%t10, %t12)
}
2024-11-21 17:03:03.583210: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !table) {
  %v1 = get_shape(%arg0)
  return(%v1)
}
2024-11-21 17:03:03.583772: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @main(%arg0: !table) {
  %v1 = ge

issue_weekday
Sunday        462992
Saturday     1108385
Monday       2488563
Wednesday    2760088
Tuesday      2809949
Friday       2891679
Thursday     2913951
Name: Summons Number, dtype: int64

## Evaluation

In [12]:
s = pd.Series([load_t, q1_t, q2_t, q3_t], index = ["data_loading", "query_1", "query_2", "query_3"])
print(f"total time taken: {s.sum()} sec")
s

total time taken: 0.4557454586029053 sec


2024-11-21 17:03:03.591961: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !pyobj, %arg1: !metadata) {
  %t2 = from_pandas.frame.metadata(%arg0, %arg1)
  %v3 = aggregate_column.scalar(%t2, 'sum')
  return(%t2, %v3)
}
2024-11-21 17:03:03.592388: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @main(%arg0: !pyobj, %arg1: !metadata) {
  %t2 = from_pandas.frame.metadata(%arg0, %arg1)
  %v3 = aggregate_column.scalar(%t2, 'sum')
  return(%t2, %v3)
}


data_loading    0.286296
query_1         0.074390
query_2         0.022752
query_3         0.072308
dtype: float64

## Let's check the number of cpu cores available for this demo

In [13]:
!nproc

128


📢 Apart from the explicit evaluation using _evaluate() method, you may also like to activate the benchmark-mode by enabling the commented code in **Cell#2** and restart and re-execute this notebook. This time the lazy-execution mode will be disabled, so each API will be executed right after it is called (as in pandas).

For the examples in this notebook there were not enough room for compiler optimization, hence you may not notice significant difference in execution time even when lazy-execution mode is disabled.

Here is my finding from the exeution when I tried:

*   Native pandas: **15 sec**
*   FireDucks.pandas without benchmark-mode (multithreaded + compiler optimized): **7.5 sec**



# Conclusion

🚀 Execution time could be reduced **from 15 sec to 7.5 sec (~2x speedup)** even for an execution environment with low spec (default colab environment seems to support only 2 cpu cores). So without incurring any migration cost (pandas to FireDucks code translation is absolutely not required) or an expensive hardware cost (no need for high spec system), you can enjoy faster analysis with FireDucks!!

🚀🚀 You may like to check other [benchmarks](https://fireducks-dev.github.io/docs/benchmarks/)

```
total time taken: 0.7562849521636963 sec
data_loading    0.517049
query_1         0.126815
query_2         0.028343
query_3         0.084078
```

```
total time taken: 6.55224871635437 sec
data_loading    1.160591
query_1         1.742037
query_2         0.587334
query_3         3.062287
```

In [14]:
df = pd.DataFrame({"a": [0, 1, 2], "b": [3, 4, 5], "c": [6, 7,8]
                })
df

,a,b,c
0,0,3,6
1,1,4,7
2,2,5,8


In [15]:
df.sort_values("a")["b"]

2024-11-21 17:03:03.817677: 2818307 fireducks/passes/fireducks_opt_mv_projection.cc:1011] Optimize Projection (fireducks.sort_values): loc("-":12:20)


2024-11-21 17:03:03.816724: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !pyobj, %arg1: !metadata) {
  %t2 = from_pandas.frame.metadata(%arg0, %arg1)
  %t3 = sort_values(%t2, ['a'], [1])
  %t4 = project(%t3, 'b')
  %v5 = get_shape(%t4)
  return(%t2, %t4, %v5)
}
2024-11-21 17:03:03.817677: 2818307 fireducks/passes/fireducks_opt_mv_projection.cc:1011] Optimize Projection (fireducks.sort_values): loc("-":12:20)
2024-11-21 17:03:03.817851: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @main(%arg0: !pyobj, %arg1: !metadata) {
  %t2 = from_pandas.frame.metadata(%arg0, %arg1)
  %t3 = project(%t2, ['b', 'a'])
  %t4 = sort_values(%t3, ['a'], [1])
  %t5 = project(%t4, 'b')
  %v6 = get_shape(%t5)
  return(%t2, %t5, %v6)
}
2024-11-21 17:03:03.823500: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !table) {
  %v1, %v2 = to_pandas.frame.metadata(%arg0)
  return(%v1, %v2)
}
2024-11-21 17:03:03.823923: 2818307 fireducks/lib/fireducks

0    3
1    4
2    5
Name: b, dtype: int64

In [16]:
import fireducks.pandas as pd
import fireducks.core
logger = fireducks.core.EvalLogger()


In [17]:
df = pd.DataFrame({"a": [0, 1, 2], "b": [3, 4, 5], "c": [6, 7,8]})
df.sort_values("a")["b"]._evaluate(_logger=logger)

2024-11-21 17:03:03.838727: 2818307 fireducks/passes/fireducks_opt_mv_projection.cc:1011] Optimize Projection (fireducks.sort_values): loc("-":12:20)


2024-11-21 17:03:03.838330: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !pyobj, %arg1: !metadata) {
  %t2 = from_pandas.frame.metadata(%arg0, %arg1)
  %t3 = sort_values(%t2, ['a'], [1])
  %t4 = project(%t3, 'b')
  return(%t4, %t2)
}
2024-11-21 17:03:03.838727: 2818307 fireducks/passes/fireducks_opt_mv_projection.cc:1011] Optimize Projection (fireducks.sort_values): loc("-":12:20)
2024-11-21 17:03:03.838864: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @main(%arg0: !pyobj, %arg1: !metadata) {
  %t2 = from_pandas.frame.metadata(%arg0, %arg1)
  %t3 = project(%t2, ['b', 'a'])
  %t4 = sort_values(%t3, ['a'], [1])
  %t5 = project(%t4, 'b')
  return(%t5, %t2)
}
2024-11-21 17:03:03.843542: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !table) {
  %v1 = get_shape(%arg0)
  return(%v1)
}
2024-11-21 17:03:03.843960: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @main(%arg0: !table) {
  %v1 = get_shape(%arg0)
 

0    3
1    4
2    5
Name: b, dtype: int64

In [18]:
x = logger.optimized_ir
f"{x}"

'module {\n  func.func @main(%arg0: !fireducks.pyobj, %arg1: !fireducks.metadata) -> (!fireducks.table, !fireducks.table, !tfrt.chain) {\n    %0 = "fire.get_string"() <{value = "a"}> : () -> !tfrt.string\n    %1 = tfrt.constant.i1 true\n    %2 = "fire.get_string"() <{value = "b"}> : () -> !tfrt.string\n    %3 = "fireducks.from_pandas.frame.metadata"(%arg0, %arg1) : (!fireducks.pyobj, !fireducks.metadata) -> !fireducks.table\n    %4 = "fireducks.make_scalar.str"(%0) : (!tfrt.string) -> !fireducks.scalar\n    %5 = "fireducks.make_column_name_element.from_scalar"(%4) : (!fireducks.scalar) -> !fireducks.column_name_element\n    %6 = "fireducks.make_column_name.from_scalar"(%5) : (!fireducks.column_name_element) -> !fireducks.column_name\n    %7 = "fireducks.make_tuple.column_name"(%6) : (!fireducks.column_name) -> tuple<!fireducks.column_name>\n    %8 = "fireducks.make_tuple.i1"(%1) : (i1) -> tuple<i1>\n    %9 = tfrt.new.chain\n    %10 = "fireducks.make_scalar.str"(%2) : (!tfrt.string) -> 

In [19]:
func @main(%arg0: !pyobj, %arg1: !metadata) {
  %t2 = from_pandas.frame.metadata(%arg0, %arg1)
  %t3 = project(%t2, ['b', 'a'])
  %t4 = sort_values(%t3, ['a'], [1])
  %t5 = project(%t4, 'b')
  %v6 = get_shape(%t5)
  return(%t2, %t5, %v6)
}


SyntaxError: invalid syntax (1760544131.py, line 1)

func @main(%arg0: !pyobj, %arg1: !metadata) {
  %t2 = from_pandas.frame.metadata(%arg0, %arg1)
  %t3 = project(%t2, ['b', 'a'])
  %t4 = sort_values(%t3, ['a'], [1])
  %t5 = project(%t4, 'b')
  %v6 = get_shape(%t5)
  return(%t2, %t5, %v6)
}


In [27]:
code = """
func @main(%arg0: !pyobj, %arg1: !metadata) {
  %t2 = from_pandas.frame.metadata(%arg0, %arg1)
  %t3 = project(%t2, ['b', 'a'])
  %t4 = sort_values(%t3, ['a'], [1])
  %t5 = project(%t4, 'b')
  %v6 = get_shape(%t5)
  return(%t2, %t5, %v6)
}
"""
print(code)


func @main(%arg0: !pyobj, %arg1: !metadata) {
  %t2 = from_pandas.frame.metadata(%arg0, %arg1)
  %t3 = project(%t2, ['b', 'a'])
  %t4 = sort_values(%t3, ['a'], [1])
  %t5 = project(%t4, 'b')
  %v6 = get_shape(%t5)
  return(%t2, %t5, %v6)
}



In [35]:
df = pd.DataFrame({"a": [0, 1, 2], "b": [3, 4, 5], "c": [6, 7,8]})
df.to_parquet("test.pq", index=False)

In [38]:
import fireducks.core
options = fireducks.core.EvalOptions()
options.set_pass_options("pushdown", True, "proj=on")
df = pd.read_parquet("test.pq").sort_values("a")["b"]._evaluate(options=options)

2024-11-21 17:24:06.482683: 2818307 fireducks/passes/fireducks_opt_mv_projection.cc:1011] Optimize Projection (fireducks.sort_values): loc("-":14:20)


2024-11-21 17:24:06.481599: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main() {
  %t0 = read_parquet('test.pq', [])
  %t1 = sort_values(%t0, ['a'], [1])
  %t2 = project(%t1, 'b')
  return(%t2)
}
2024-11-21 17:24:06.482683: 2818307 fireducks/passes/fireducks_opt_mv_projection.cc:1011] Optimize Projection (fireducks.sort_values): loc("-":14:20)
2024-11-21 17:24:06.483017: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @main() {
  %t0 = read_parquet('test.pq', [])
  %t1 = project(%t0, ['b', 'a'])
  %t2 = sort_values(%t1, ['a'], [1])
  %t3 = project(%t2, 'b')
  return(%t3)
}


In [39]:
df

2024-11-21 17:24:12.062391: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !table) {
  %v1 = get_shape(%arg0)
  return(%v1)
}
2024-11-21 17:24:12.063401: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @main(%arg0: !table) {
  %v1 = get_shape(%arg0)
  return(%v1)
}
2024-11-21 17:24:12.064505: 2818307 fireducks/lib/fireducks_core.cc:64] Input IR:
func @main(%arg0: !table) {
  %v1, %v2 = to_pandas.frame.metadata(%arg0)
  return(%v1, %v2)
}
2024-11-21 17:24:12.065436: 2818307 fireducks/lib/fireducks_core.cc:73] Optimized IR:
func @main(%arg0: !table) {
  %v1, %v2 = to_pandas.frame.metadata(%arg0)
  return(%v1, %v2)
}


0    3
1    4
2    5
Name: b, dtype: int64